# 4 · Guainía: polarimetric analysis

**Pipeline:** 0 · Data access → 1 · Stack → 2 · Tomography → 3 · Forest height → **4 · Polarimetric analysis**

Two complementary analyses of the quad-pol stack:

**Part A, single-acquisition polarimetry (2-D).** The reference acquisition is calibrated to σ⁰ with the `/radiometry` LUT, and the thermal noise from the `/denoising` LUT is removed from the coherency matrix $\mathbf T_3$. It is then decomposed with PolSARpro:
backscatter and noise level → Pauli RGB → Cloude–Pottier $H/A/\alpha$ and the $H$–$\alpha$ plane → Freeman–Durden → Wishart $H/A/\alpha$ classification → temporal stability over the 7 acquisitions.

**Part B, polarimetric tomography (3-D).** The four phase-corrected channels are focused **coherently** at each height with one common filter, so that $\mathbf S(z) = [S_{HH}, S_{HV}, S_{VH}, S_{VV}](z)$ is a valid scattering matrix:
$$S_{pq}(z) = \mathbf w^H(z)\,\mathbf y_{pq},\qquad \mathbf w = \mathbf a(z)/N\ \text{(beamforming)}\quad\text{or}\quad \frac{\mathbf R^{-1}\mathbf a}{\mathbf a^H\mathbf R^{-1}\mathbf a}\ \text{(Capon)}.$$
Every height slice is decomposed (Pauli, $H/A/\alpha$, Freeman–Durden). The phase conventions come from notebook 2 and the forest height from notebook 3.

PolSARpro layout throughout: `y` = azimuth, `x` = range. Maps are drawn at true ground scale. x is ground range and y is azimuth, with the first line at the top, which for this descending pass is roughly north-up. Colours are stretched between the 2nd and 98th percentiles in dB, so that the small contrasts inside closed forest stay visible.

**GeoTIFFs** (like notebook 3, section 10): the Part A Pauli RGB is exported as an 8-bit RGB GeoTIFF for GIS display, plus a 3-band dB GeoTIFF with the values. Part B exports a Pauli RGB GeoTIFF for each height in `GEOTIFF_HEIGHTS`.

## 1. Configuration
Polarimetric processing runs at full resolution, so it uses a block of the analysis crop (`POL_CROP`, stack pixel indices). The default is 4000 azimuth lines (≈ 27 km) at the crop centre, over the full range.
`BOX` is the multilook window (azimuth × range samples). (21, 3) ≈ 140 m × 140 m on the ground, giving 63 looks.

In [ ]:
import pathlib, sys
SITE_DIR = pathlib.Path.cwd()                       # the Guainia/ folder (notebooks run from here)
if not (SITE_DIR / "stack_download").is_dir():
    raise FileNotFoundError("open the notebook from the Guainia/ folder (stack_download/ not found)")
ROOT = next(p for p in [SITE_DIR, *SITE_DIR.parents] if (p / "biomass_tomo").is_dir())
sys.path.insert(0, str(ROOT))                       # the biomass_tomo package

import time
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

from biomass_tomo import stack as st, polsar, poltomo, reference as ref, display as dp

MANIFEST = SITE_DIR / "Guainia_outputs" / "run_manifest.json"
POL_CROP = None                       # (az0, az1, rg0, rg1) in stack indices; None = 4000 central lines of the analysis crop,
                                      # range limited to samples with data (no-data frame edges bias the Wishart classes)
POL_LINES = 4000
BOX = (21, 3)                         # multilook (azimuth, range) samples
STEP = (BOX[0] // 3, 1)               # output decimation for Part A maps
CALIBRATION = "sigma0"                # "sigma0", "gamma0" or "beta0"
NOISE_REMOVAL = True
NOISE_TOL_DB = 1.5                    # noise-LUT check: tolerance around the across-stack median

# Part B
HEIGHTS = np.arange(-10, 50.1, 2.5)   # focusing heights above ground [m]
FOCUSING = "beamforming"              # "beamforming" or "capon"
CAPON_WINDOW_M, CAPON_LOAD = 150.0, 0.01
MAP_HEIGHTS = [0, 10, 20, 30, 40]
STRETCH = (2, 98)                     # percentile colour stretch (dB for powers)
GEOTIFF_HEIGHTS = [0, 20]             # Part B: heights exported as Pauli RGB GeoTIFFs ([] = none)

cfg = st.read_manifest(MANIFEST)
if "phase" not in cfg:
    raise RuntimeError("run notebook 2 first: the phase conventions come from it")
ph = cfg["phase"]
crop = cfg["crop"]
if POL_CROP is None:
    c = (crop[0] + crop[1]) // 2
    ds = st.open_stack(cfg["stack_path"])
    line = ds.slc_real.sel(polarisation="HH").isel(acquisition=ph["reference_index"], azimuth=slice(c - 50, c + 50),
                                                   range=slice(crop[2], crop[3])).values
    has = np.flatnonzero((line != 0).all(axis=0))
    POL_CROP = [max(crop[0], c - POL_LINES // 2), min(crop[1], c + POL_LINES // 2), crop[2] + int(has[0]), crop[2] + int(has[-1]) + 1]
OUT = pathlib.Path(cfg["out_dir"]) / "polarimetry"
OUT.mkdir(parents=True, exist_ok=True)
blk = f"az{POL_CROP[0]}-{POL_CROP[1]}_rg{POL_CROP[2]}-{POL_CROP[3]}"
print("polarimetry block (az0, az1, rg0, rg1):", POL_CROP, "| phase conventions:", ph)

## 2. Data
The block is loaded once for all acquisitions and reordered to (azimuth, range[, acquisition]).

In [ ]:
t0 = time.time()
ds = st.open_stack(cfg["stack_path"])
aux_names = ("dem_phases", "calibration_phases", "insar_wavenumbers", "incidence_angle", "latitude", "longitude",
             "sigma_nought_factor", "gamma_nought_factor", "noise_HH", "noise_HV", "noise_VH", "noise_VV")
data = st.load_crop(ds, POL_CROP, aux_names=aux_names)
to_yx = lambda a: np.ascontiguousarray(np.swapaxes(a, 0, 1))
aux = {k: to_yx(v) for k, v in data["aux"].items()}
slc = {p: to_yx(v) for p, v in data["slc"].items()}
acq = data["acquisition"]
d_az = float(ds.azimuth[1] - ds.azimuth[0]); d_rg = float(ds.range[1] - ds.range[0])
theta = np.deg2rad(np.nanmean(aux["incidence_angle"]))
REF = ph["reference_index"]
Ny, Nx, Nb = aux["insar_wavenumbers"].shape
del data
az_km = (POL_CROP[0] + np.arange(Ny)) * d_az / 1e3                      # along-track position [km]
gr_km = dp.ground_range_km(ds.range.values[POL_CROP[2]:POL_CROP[3]], aux["incidence_angle"].T)
print(f"{Ny} azimuth x {Nx} range samples, {Nb} acquisitions, loaded in {time.time() - t0:.0f} s; "
      f"block {gr_km[-1]:.0f} km (ground range) x {az_km[-1] - az_km[0]:.0f} km (azimuth)")
print(f"multilook {BOX[0]} x {BOX[1]} = {BOX[0] * d_az:.0f} m x {BOX[1] * d_rg / np.sin(theta):.0f} m ground, {BOX[0] * BOX[1]} looks")

# Part A: single-acquisition polarimetry

## A1. Calibration and noise
The L1c SLC is β⁰. Power is converted with the `/radiometry` LUT factors, σ⁰ = β⁰·`sigma_nought_factor` (≈ sin θ) and γ⁰ = β⁰·`gamma_nought_factor` (≈ tan θ). The `/denoising` LUT gives the thermal-noise power per channel (β⁰); it is **not** removed in L1c. At P band the cross-pol is only a few dB above the noise, so noise inflates HV and pushes entropy towards 1. Noise is subtracted from the $\mathbf T_3$ diagonal and from $\mathrm{Re}\,T_{12}$:
$$T_{11},T_{22} \mathrel{-}= \tfrac12(N_{HH}+N_{VV}),\quad \mathrm{Re}\,T_{12} \mathrel{-}= \tfrac12(N_{HH}-N_{VV}),\quad T_{33} \mathrel{-}= \tfrac12(N_{HV}+N_{VH}).$$

**Noise-LUT check.** The noise LUT is not reliable in every product. In this stack, C02 reports HH/HV noise at about −4 dB, above the measured signal, although its backscatter matches the other dates. An acquisition whose median noise departs from the across-stack median by more than `NOISE_TOL_DB` in any channel is flagged. Its noise is replaced by the per-pixel median over the unflagged acquisitions.

In [ ]:
factor = {"sigma0": aux["sigma_nought_factor"], "gamma0": aux["gamma_nought_factor"], "beta0": None}[CALIBRATION]
levels = {p: 10 * np.log10(np.nanmedian(aux[f"noise_{p}"][::20, ::10], axis=(0, 1))) for p in polsar.POLS}
print(pd.DataFrame(levels, index=[f"C{int(a):02d}" for a in acq]).round(2).T.to_string())
BAD_NOISE = polsar.noise_check(levels, NOISE_TOL_DB)
good = [i for i in range(Nb) if i not in BAD_NOISE]
for p in polsar.POLS:
    if BAD_NOISE:
        aux[f"noise_{p}"][:, :, BAD_NOISE] = np.median(aux[f"noise_{p}"][:, :, good], axis=2, keepdims=True)
print("noise LUT flagged and replaced for:", [f"C{int(acq[i]):02d}" for i in BAD_NOISE] or "none")
noise = {p: aux[f"noise_{p}"][:, :, REF] for p in polsar.POLS}
ref_slc = {p: slc[p][:, :, REF] for p in polsar.POLS}
t0 = time.time()
T3_raw = polsar.coherency_T3(ref_slc, factor=factor, noise=None, box=BOX)
T3 = polsar.coherency_T3(ref_slc, factor=factor, noise=noise if NOISE_REMOVAL else None, box=BOX)
print(f"T3 in {time.time() - t0:.0f} s")
dec = (slice(STEP[0] // 2, None, STEP[0]), slice(STEP[1] // 2, None, STEP[1]))
T3d, T3d_raw = T3.isel(y=dec[0], x=dec[1]).persist(), T3_raw.isel(y=dec[0], x=dec[1]).persist()
lat_d, lon_d, inc_d = aux["latitude"][dec], aux["longitude"][dec], aux["incidence_angle"][dec]
valid_d = (np.abs(ref_slc["HH"]) > 0)[dec]
az_d, gr_d = az_km[dec[0]], gr_km[dec[1]]
show = lambda ax, img, **kw: dp.show(ax, img, az_d, gr_d, **kw)          # Part A maps (y, x)
bs_raw, bs = polsar.backscatter(T3d_raw), polsar.backscatter(T3d)
nesz = {p: (noise[p] * (factor if factor is not None else 1))[dec] for p in polsar.POLS}

In [ ]:
db = dp.to_db                         # dB; powers <= -100 dB (noise-clipped, no-data) -> NaN
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), constrained_layout=True)
inc_bins = np.linspace(np.nanmin(inc_d), np.nanmax(inc_d), 25); centres = 0.5 * (inc_bins[1:] + inc_bins[:-1])
for p, c in (("HH", "tab:blue"), ("VV", "tab:green"), ("XX", "tab:red")):
    v = bs_raw[p]
    axes[0].plot(centres, [np.nanmedian(db(v)[valid_d & (inc_d >= a) & (inc_d < b)]) for a, b in zip(inc_bins[:-1], inc_bins[1:])], color=c, label=f"{p} (with noise)")
axes[0].plot(centres, [np.nanmedian(db(nesz["HV"])[(inc_d >= a) & (inc_d < b)]) for a, b in zip(inc_bins[:-1], inc_bins[1:])], "k--", label="noise HV (NESZ)")
axes[0].set_xlabel("incidence [deg]"); axes[0].set_ylabel(f"{CALIBRATION} [dB]"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[0].set_title("median backscatter and noise level vs incidence", fontsize=9)
for p, c in (("HH", "tab:blue"), ("VV", "tab:green"), ("XX", "tab:red")):
    axes[1].hist(db(bs[p])[valid_d].ravel(), bins=120, range=(-30, 5), histtype="step", color=c, density=True, label=f"{p}, noise removed")
    axes[1].hist(db(bs_raw[p])[valid_d].ravel(), bins=120, range=(-30, 5), histtype="step", color=c, ls=":", density=True)
axes[1].set_xlabel(f"{CALIBRATION} [dB]"); axes[1].legend(fontsize=8); axes[1].set_title("distributions (dotted: with noise)", fontsize=9)
plt.show()
snr = np.nanmedian(db(bs_raw["XX"])[valid_d] - db(nesz["HV"])[valid_d])
print(f"median cross-pol SNR {snr:.1f} dB")

## A2. Backscatter maps and Pauli RGB
σ⁰ in HH, cross-pol (XX = (HV+VH)/2) and VV, each stretched between its own 2nd and 98th percentiles. The Pauli RGB uses R $T_{22}$ (double bounce, $|HH-VV|^2$), G $T_{33}$ (volume, $|HV+VH|^2$) and B $T_{11}$ (surface, $|HH+VV|^2$), each channel stretched in dB. Forest is green, ground–trunk interaction (flooded forest, river banks) red/magenta, and smooth surfaces and water blue/dark.

In [ ]:
T11, T22, T33 = (np.where(valid_d, T3d[m].values, np.nan) for m in ("m11", "m22", "m33"))
pauli_rgb = dp.rgb_db(T22, T33, T11, STRETCH)
fig, axes = plt.subplots(2, 2, figsize=(15, 7.8), sharex=True, sharey=True, constrained_layout=True)
for ax, p in zip(axes.flat, ("HH", "XX", "VV")):
    v = np.where(valid_d, db(bs[p]), np.nan)
    lo, hi = dp.limits(v, STRETCH)
    im = show(ax, v, cmap="gray", vmin=lo, vmax=hi)
    ax.set_title(f"{CALIBRATION} {p} [dB]", fontsize=9); fig.colorbar(im, ax=ax, shrink=0.8)
show(axes[1, 1], pauli_rgb); axes[1, 1].set_title("Pauli RGB (R: T22, G: T33, B: T11)", fontsize=9)
for ax in axes[:, 0]: ax.set_ylabel("azimuth [km]")
plt.show()

## A2b. Pauli RGB GeoTIFF
As in notebook 3 (section 10), the Pauli image is placed on a regular lon/lat grid (EPSG:4326) by nearest neighbour from the LUT geolocation. Two files are written:
- `pauli_rgb_<block>.tif`: 8-bit RGB with the same dB stretch as above (0 = no-data), for direct display in QGIS or Google Earth;
- `pauli_db_<block>.tif`: three float bands $T_{22}$, $T_{33}$, $T_{11}$ in dB (σ⁰ units), for analysis or a different stretch.

The preview below is read back from the RGB GeoTIFF.

In [ ]:
import rasterio
pauli_tif = OUT / f"pauli_rgb_{blk}.tif"
rgb8 = dp.rgb_uint8(pauli_rgb, valid_d)
_, res = ref.to_geotiff({"R_T22": rgb8[..., 0], "G_T33": rgb8[..., 1], "B_T11": rgb8[..., 2]}, lat_d, lon_d, pauli_tif, rgb=True)
pauli_db_tif = OUT / f"pauli_db_{blk}.tif"
ref.to_geotiff({"T22_dB": db(T22), "T33_dB": db(T33), "T11_dB": db(T11)}, lat_d, lon_d, pauli_db_tif)
print(f"{pauli_tif.name} and {pauli_db_tif.name}: {res:.5f} deg ≈ {res * 111e3:.0f} m")
with rasterio.open(pauli_tif) as src:
    img, b = src.read().transpose(1, 2, 0), src.bounds
fig, ax = plt.subplots(figsize=(9, 8), constrained_layout=True)
ax.imshow(np.where(img.any(axis=-1, keepdims=True), img, 255), extent=[b.left, b.right, b.bottom, b.top], interpolation="none")
ax.set_aspect(1 / np.cos(np.deg2rad(np.nanmean(lat_d)))); ax.set_xlabel("longitude [deg]"); ax.set_ylabel("latitude [deg]")
ax.set_title(f"Pauli RGB GeoTIFF (EPSG:4326), C{int(acq[REF]):02d}", fontsize=10)
plt.show()

## A3. Cloude–Pottier $H/A/\alpha$
The eigen-decomposition of $\mathbf T_3$ gives the **entropy** $H$ (0 = one mechanism, 1 = random), the **anisotropy** $A$ (relative weight of the 2nd and 3rd mechanisms) and the mean **α** angle (0° surface, 45° dipole/volume, 90° double bounce). Tropical forest at P band is expected in zone Z5/Z2 (high entropy, α ≈ 40–55°). The plane is shown with and without noise removal.

In [ ]:
from polsarpro.decompositions import h_a_alpha, freeman
t0 = time.time()
haa = h_a_alpha(T3d, boxcar_size=[1, 1], flags=("entropy", "anisotropy", "alpha")).compute()
haa_raw = h_a_alpha(T3d_raw, boxcar_size=[1, 1], flags=("entropy", "anisotropy", "alpha")).compute()
print(f"H/A/alpha in {time.time() - t0:.0f} s")
fig, axes = plt.subplots(2, 2, figsize=(15, 7.8), sharex=True, sharey=True, constrained_layout=True)
for ax, (n, cmap) in zip(axes.flat, (("entropy", "viridis"), ("anisotropy", "cividis"), ("alpha", "plasma"))):
    v = np.where(valid_d, haa[n].values, np.nan)
    lo, hi = dp.limits(v, STRETCH)
    im = show(ax, v, cmap=cmap, vmin=lo, vmax=hi)
    ax.set_title(f"{n} (stretch {lo:.2f}–{hi:.2f})", fontsize=9); fig.colorbar(im, ax=ax, shrink=0.8)
v = np.where(valid_d, haa_raw.entropy.values - haa.entropy.values, np.nan)
im = show(axes[1, 1], v, cmap="magma", vmin=0, vmax=dp.limits(v, STRETCH)[1])
axes[1, 1].set_title("entropy increase caused by noise", fontsize=9); fig.colorbar(im, ax=axes[1, 1], shrink=0.8)
for ax in axes[:, 0]: ax.set_ylabel("azimuth [km]")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True, gridspec_kw={"width_ratios": [1.3, 1]})
H, Al = haa.entropy.values[valid_d], haa.alpha.values[valid_d]
axes[0].hist2d(H, Al, bins=[200, 180], range=[[0, 1], [0, 90]], cmap="magma_r", cmin=1, norm="log")
polsar.h_alpha_plane(axes[0]); axes[0].set_title(f"H–α plane (log density), noise {'removed' if NOISE_REMOVAL else 'kept'}", fontsize=9)
axes[1].hist(haa_raw.entropy.values[valid_d], bins=200, range=(0, 1), histtype="step", label="with noise")
axes[1].hist(H, bins=200, range=(0, 1), histtype="step", label="noise removed")
axes[1].set_xlim(*dp.limits([H, haa_raw.entropy.values[valid_d]], (0.5, 99.9))); axes[1].set_xlabel("entropy H"); axes[1].legend(fontsize=8)
plt.show()

## A4. Freeman–Durden
Three-component model-based decomposition into surface ($P_s$), double-bounce ($P_d$) and volume ($P_v$) power. Shown as an RGB and as fractions of the total.

In [ ]:
fd = freeman(T3d, boxcar_size=[1, 1]).compute()
Ps, Pd, Pv = (np.clip(fd[k].values, 0, None) for k in ("odd", "double", "volume"))
tot = Ps + Pd + Pv
m = lambda v: np.where(valid_d & (tot > 0), v, np.nan)
fig, axes = plt.subplots(2, 2, figsize=(15, 7.8), sharex=True, sharey=True, constrained_layout=True)
show(axes[0, 0], dp.rgb_db(m(Pd), m(Pv), m(Ps), STRETCH)); axes[0, 0].set_title("Freeman–Durden RGB (R: Pd, G: Pv, B: Ps)", fontsize=9)
for ax, (n, v, cmap) in zip([axes[0, 1], axes[1, 0], axes[1, 1]],
                            (("Pv / total", Pv, "Greens"), ("Ps / total", Ps, "Blues"), ("Pd / total", Pd, "Reds"))):
    f_ = m(v / tot)
    lo, hi = dp.limits(f_, STRETCH)
    im = show(ax, f_, cmap=cmap, vmin=lo, vmax=hi); ax.set_title(f"{n} (stretch {lo:.2f}–{hi:.2f})", fontsize=9)
    fig.colorbar(im, ax=ax, shrink=0.8)
for ax in axes[:, 0]: ax.set_ylabel("azimuth [km]")
plt.show()
fr = pd.DataFrame({k: np.nanpercentile((v / tot)[valid_d], [10, 50, 90]) for k, v in (("Ps", Ps), ("Pd", Pd), ("Pv", Pv))},
                  index=["p10", "median", "p90"])
fr.round(3)

## A5. Wishart $H/A/\alpha$ classification
Unsupervised: an initial partition by the $H$–$\alpha$ zones, refined by iterations of the complex Wishart distance to the class centres, then split into 16 classes by anisotropy. Classes are data-driven; check their mean $H$, α and Pauli powers before interpreting them.

In [ ]:
from polsarpro.classification import wishart_h_a_alpha
t0 = time.time()
cls = wishart_h_a_alpha(T3d, boxcar_size=[1, 1], max_iter=10, verbose=False).compute()
print(f"Wishart in {time.time() - t0:.0f} s")
labels = cls["wishart_h_a_alpha_class"].values.astype(float)
labels[~valid_d] = np.nan
fig, ax = plt.subplots(figsize=(10, 5.2), constrained_layout=True)
im = show(ax, labels, cmap="tab20", vmin=0.5, vmax=16.5)
fig.colorbar(im, ax=ax, shrink=0.8, label="class", ticks=range(1, 17)); ax.set_ylabel("azimuth [km]")
ax.set_title("Wishart H/A/α classes", fontsize=9)
plt.show()
summary = pd.DataFrame([{"class": int(c), "fraction_%": (labels == c).mean() * 100,
                         "H": np.nanmean(haa.entropy.values[labels == c]), "A": np.nanmean(haa.anisotropy.values[labels == c]),
                         "alpha": np.nanmean(haa.alpha.values[labels == c]),
                         "Pv_frac": np.nanmean((Pv / tot)[labels == c]), "XX_dB": np.nanmedian(db(bs["XX"])[labels == c])}
                        for c in np.unique(labels[np.isfinite(labels)])]).set_index("class")
summary.round(2)

## A6. Temporal stability over the 7 acquisitions
Median σ⁰ per channel (noise removed, with the checked noise of section A1) and the cross-pol ratio for every acquisition, over the block. Over 18 days in the dry-to-wet transition, changes of more than ~0.5 dB point to rain or soil-moisture events. Those also show up as temporal decorrelation in the tomography.

In [ ]:
rows = []
f = factor if factor is not None else 1.0
for i in range(Nb):
    med = {}
    for p in polsar.POLS:
        I = np.abs(slc[p][:, :, i]) ** 2
        ok = I > 0
        med[p] = 10 * np.log10(np.median(((I - (aux[f"noise_{p}"][:, :, i] if NOISE_REMOVAL else 0)) * f)[ok][::7]))
    rows.append({"cycle": f"C{int(acq[i]):02d}", "date": str(ds.product_id.values[i])[15:23], **{f"{p}_dB": med[p] for p in polsar.POLS},
                 "HV/HH_dB": med["HV"] - med["HH"]})
temporal = pd.DataFrame(rows).set_index("cycle")
fig, ax = plt.subplots(figsize=(7, 3.2), constrained_layout=True)
for p in polsar.POLS:
    ax.plot(temporal.index, temporal[f"{p}_dB"], "o-", label=p)
ax.set_ylabel(f"median {CALIBRATION} [dB]"); ax.legend(ncol=4, fontsize=8); ax.grid(alpha=0.3)
plt.show()
temporal.round(2)

# Part B: polarimetric tomography

## B1. Phase correction and vertical focusing
The same LUT corrections and signs as in notebook 2 are applied to every acquisition. The four channels are then focused at each height in `HEIGHTS`. The boxcar for the per-height decompositions is `BOX`, with one output per window. Power-ratio parameters ($H$, α, fractions) are unaffected by the per-pixel calibration factor. Noise is not removed here.

In [ ]:
corr = np.exp(1j * (ph["sign_dem"] * aux["dem_phases"] + ph["sign_cal"] * aux["calibration_phases"])).astype(np.complex64)
for p in slc:
    slc[p] *= corr
del corr
kz = (ph["kz_sign"] * aux["insar_wavenumbers"]).astype(np.float32)
t0 = time.time()
cube, (sy, sx) = poltomo.polarimetric_cube(
    slc, kz, HEIGHTS, boxcar=BOX, step=BOX, focusing=FOCUSING,
    capon_kw=dict(window_m=CAPON_WINDOW_M, d_az=d_az, d_rg=d_rg, theta=theta, d_load=CAPON_LOAD),
    progress=lambda k, z: print(f"\rz = {int(z):+3d} m ({k + 1}/{len(HEIGHTS)})", end="", flush=True))
print(f"\nfocused and decomposed in {time.time() - t0:.0f} s; cube {cube['T11'].shape} (height, y, x)")
lat_s, lon_s = aux["latitude"][sy, sx], aux["longitude"][sy, sx]
valid_s = (np.abs(slc["HH"][:, :, REF]) > 0)[sy, sx]
az_s, gr_s = az_km[sy][:cube["T11"].shape[1]], gr_km[sx][:cube["T11"].shape[2]]

## B2. Vertical profiles
Median and interquartile range of each parameter against height. At P band one expects ground and ground–trunk double bounce near $z = 0$ (higher $T_{22}$/span, $P_d$, larger α), random volume in the canopy ($T_{33}$, $P_v$, $H \to 1$, α ≈ 45°), and only side-lobe leakage above the canopy top.

In [ ]:
panels = {"Pauli fraction": {"T11/span": "T11_frac", "T22/span": "T22_frac", "T33/span": "T33_frac"},
          "Freeman fraction": {"Ps": "Ps_frac", "Pd": "Pd_frac", "Pv": "Pv_frac"},
          "entropy / anisotropy": {"H": "entropy", "A": "anisotropy"},
          "alpha [deg]": {"alpha": "alpha"},
          "span [dB]": {"span": "span"}}
colours = ["tab:blue", "tab:red", "tab:green"]
fig, axes = plt.subplots(1, len(panels), figsize=(3.4 * len(panels), 4.5), sharey=True, constrained_layout=True)
for ax, (title, items) in zip(axes, panels.items()):
    for c, (lab, name) in zip(colours, items.items()):
        v = cube[name][:, valid_s]
        if name == "span":
            v = 10 * np.log10(v)
        q = np.nanpercentile(v, [25, 50, 75], axis=1)
        ax.plot(q[1], HEIGHTS, color=c, label=lab); ax.fill_betweenx(HEIGHTS, q[0], q[2], color=c, alpha=0.15)
    ax.axhline(0, color="k", lw=0.8); ax.set_title(title, fontsize=9); ax.legend(fontsize=8); ax.grid(alpha=0.3)
axes[0].set_ylabel("height above ground [m]")
plt.show()

## B3. Height slices
Pauli RGB, Freeman RGB, entropy and α at selected heights. Each RGB channel has one scaling shared by all heights, so differences between slices reflect vertical structure.

In [ ]:
idx = [int(np.argmin(np.abs(HEIGHTS - h))) for h in MAP_HEIGHTS]
vm = lambda n: np.where(valid_s[None], cube[n], np.nan)
rgbs = {"Pauli RGB": dp.rgb_db(vm("T22"), vm("T33"), vm("T11"), STRETCH),
        "Freeman RGB": dp.rgb_db(vm("Pd"), vm("Pv"), vm("Ps"), STRETCH),
        "entropy H": vm("entropy"), "alpha": vm("alpha")}
fig, axes = plt.subplots(len(rgbs), len(idx), figsize=(3.7 * len(idx), 2.15 * len(rgbs) + 0.6),
                         sharex=True, sharey=True, constrained_layout=True)
for row, (title, vol) in enumerate(rgbs.items()):
    if vol.ndim == 3:
        lo, hi = dp.limits([vol[k] for k in idx], STRETCH)          # one scale for all heights
    for col, k in enumerate(idx):
        ax = axes[row, col]
        if vol.ndim == 4:
            dp.show(ax, vol[k], az_s, gr_s)
        else:
            im = dp.show(ax, vol[k], az_s, gr_s, cmap="viridis" if title.startswith("entropy") else "plasma", vmin=lo, vmax=hi)
        ax.set_xlabel("")
        if row == 0: ax.set_title(f"z = {HEIGHTS[k]} m", fontsize=10)
        if col == 0: ax.set_ylabel(f"{title}\nazimuth [km]", fontsize=9)
    if vol.ndim == 3:
        fig.colorbar(im, ax=axes[row, :], shrink=0.8, pad=0.01, label=f"{lo:.2f}–{hi:.2f}")
for ax in axes[-1]: ax.set_xlabel("ground range [km]")
plt.show()

slice_tifs = []
for h in GEOTIFF_HEIGHTS:
    k = int(np.argmin(np.abs(HEIGHTS - h)))
    rgb8 = dp.rgb_uint8(rgbs["Pauli RGB"][k], valid_s)
    t = OUT / f"pauli_rgb_z{int(HEIGHTS[k]):+03d}m_{FOCUSING}_{blk}.tif"
    ref.to_geotiff({"R_T22": rgb8[..., 0], "G_T33": rgb8[..., 1], "B_T11": rgb8[..., 2]}, lat_s, lon_s, t, rgb=True)
    slice_tifs.append(str(t))
print("Pauli RGB GeoTIFFs of height slices:", *[pathlib.Path(t).name for t in slice_tifs], sep="\n  ")

## B4. Vertical section with forest height
Pauli RGB along azimuth at the middle range of the block, averaged over 5 range cells and stretched on the section itself (2.5 m height steps, bilinear display), with the heights from notebook 3 matched by geolocation: canopy top (white), bias-corrected canopy top (yellow), the reference `ref_mean` (cyan, dashed) and the phase centre (white, dotted).

In [ ]:
xc = cube["T11"].shape[2] // 2
sec = {n: np.nanmean(vm(n)[:, :, max(xc - 2, 0):xc + 3], axis=2) for n in ("T22", "T33", "T11")}
section = dp.rgb_db(sec["T22"], sec["T33"], sec["T11"], STRETCH)  # stretched on the section itself
sec_km = az_s
dz = HEIGHTS[1] - HEIGHTS[0]
fig, ax = plt.subplots(figsize=(14, 5), constrained_layout=True)
ax.imshow(section, aspect="auto", origin="lower", interpolation="bilinear",
          extent=[sec_km[0], sec_km[-1], HEIGHTS[0] - dz / 2, HEIGHTS[-1] + dz / 2])
if "forest_height" in cfg and pathlib.Path(cfg["forest_height"]["heights"]).exists():
    from scipy.spatial import cKDTree
    with xr.open_dataset(cfg["forest_height"]["heights"]) as f_:
        Hh = f_.load()                                       # load and release the file (no lock left open)
    ok = np.isfinite(Hh.latitude.values.ravel())
    tree = cKDTree(np.column_stack([Hh.latitude.values.ravel()[ok], Hh.longitude.values.ravel()[ok]]))
    _, nn = tree.query(np.column_stack([lat_s[:, xc], lon_s[:, xc]]))
    for name, style in (("h_canopy_top", "w-"), ("h_canopy_top_corrected", "y-"), ("ref_mean", "c--"), ("z_phase_centre", "w:")):
        if name in Hh:
            ax.plot(sec_km, Hh[name].values.ravel()[ok][nn], style, lw=1.4, label=name)
    ax.legend(fontsize=8, loc="upper right")
ax.axhline(0, color="w", ls="--", lw=0.6)
ax.set_xlabel("azimuth [km]"); ax.set_ylabel("height [m]"); ax.set_title("Pauli RGB section (R: T22, G: T33, B: T11)", fontsize=9)
plt.show()

## B5. Polarimetry vs forest height
Ground-level double-bounce fraction and canopy-level volume fraction and entropy against the canopy-top height of notebook 3. This tests whether the vertical polarimetric signature follows forest structure.

In [ ]:
if "forest_height" in cfg:
    h_top = Hh.h_canopy_top.values.ravel()[ok][tree.query(np.column_stack([lat_s.ravel(), lon_s.ravel()]))[1]].reshape(lat_s.shape)
    k0, k20 = int(np.argmin(np.abs(HEIGHTS - 0))), int(np.argmin(np.abs(HEIGHTS - 20)))
    feats = {"T22/span at z=0": cube["T22_frac"][k0], "Pv fraction at z=20 m": cube["Pv_frac"][k20],
             "entropy at z=20 m": cube["entropy"][k20], "alpha at z=0": cube["alpha"][k0]}
    fig, axes = plt.subplots(1, len(feats), figsize=(3.4 * len(feats), 3.4), constrained_layout=True)
    for ax, (lab, v) in zip(axes, feats.items()):
        m = valid_s & np.isfinite(v) & np.isfinite(h_top)
        ax.hist2d(h_top[m], v[m], bins=40, cmap="Greys", cmin=1)
        r = np.corrcoef(h_top[m], v[m])[0, 1] if m.sum() > 3 else np.nan
        ax.set_xlabel("h_canopy_top [m]"); ax.set_ylabel(lab, fontsize=9); ax.set_title(f"r = {r:.2f}, n = {m.sum()}", fontsize=9)
    plt.show()
else:
    print("run notebook 3 for the forest height")

## Saving
- `polarimetry/polsar_<block>.nc`: Part A maps (σ⁰ per channel, $\mathbf T_3$ diagonal, $H/A/\alpha$, Freeman–Durden, Wishart class, geolocation) on the decimated grid.
- `polarimetry/poltomo_<focusing>_<block>.nc`: the Part B cube (height, y, x).
- GeoTIFFs written above: `pauli_rgb_<block>.tif`, `pauli_db_<block>.tif` and `pauli_rgb_z<height>m_<focusing>_<block>.tif`.

Both are recorded in the manifest under `polarimetry`.

In [ ]:
yx = ("y", "x")
A = xr.Dataset({**{f"{CALIBRATION}_{p}": (yx, bs[p].astype(np.float32)) for p in bs},
                **{f"T{k}{k}": (yx, T3d[f"m{k}{k}"].values.astype(np.float32)) for k in (1, 2, 3)},
                **{n: (yx, haa[n].values.astype(np.float32)) for n in ("entropy", "anisotropy", "alpha")},
                "Ps": (yx, Ps.astype(np.float32)), "Pd": (yx, Pd.astype(np.float32)), "Pv": (yx, Pv.astype(np.float32)),
                "wishart_class": (yx, labels.astype(np.float32)), "valid": (yx, valid_d.astype(np.int8)),
                "latitude": (yx, lat_d.astype(np.float32)), "longitude": (yx, lon_d.astype(np.float32))},
               coords={"y": POL_CROP[0] + np.arange(T3d.sizes["y"]) * STEP[0] + STEP[0] // 2,
                       "x": POL_CROP[2] + np.arange(T3d.sizes["x"]) * STEP[1] + STEP[1] // 2},
               attrs={"stack": cfg["stack_path"], "acquisition": f"C{int(acq[REF]):02d}", "calibration": CALIBRATION,
                      "noise_removed": int(NOISE_REMOVAL), "boxcar": str(BOX), "step": str(STEP),
                      "layout": "y = azimuth line, x = range sample (stack indices)"})
pathA = OUT / f"polsar_{blk}.nc"
st.save_netcdf(A, pathA)

B = xr.Dataset({k: (("height", "y", "x"), v.astype(np.float32)) for k, v in cube.items()},
               coords={"height": HEIGHTS, "y": POL_CROP[0] + np.arange(cube["T11"].shape[1]) * BOX[0] + BOX[0] // 2,
                       "x": POL_CROP[2] + np.arange(cube["T11"].shape[2]) * BOX[1] + BOX[1] // 2})
B["latitude"] = (("y", "x"), lat_s.astype(np.float32)); B["longitude"] = (("y", "x"), lon_s.astype(np.float32))
B.attrs = {"stack": cfg["stack_path"], "focusing": FOCUSING, "boxcar": str(BOX), **{k: str(v) for k, v in ph.items()},
           "layout": "y = azimuth line, x = range sample (stack indices)"}
pathB = OUT / f"poltomo_{FOCUSING}_{blk}.nc"
st.save_netcdf(B, pathB)

cfg["polarimetry"] = {"block": POL_CROP, "polsar": str(pathA), "poltomo": str(pathB), "calibration": CALIBRATION,
                      "noise_removed": NOISE_REMOVAL, "focusing": FOCUSING,
                      "pauli_geotiff": str(pauli_tif), "pauli_db_geotiff": str(pauli_db_tif), "pauli_slice_geotiffs": slice_tifs}
st.write_manifest(MANIFEST, **cfg)
print("saved", pathA, "and", pathB)